# 02 · Robustness Experiments — the dose–response curve, the prediction-line gap, and the trimmed-loss defense

This notebook runs the pre-registered experiment: train the compact separator at each bleed
level, assemble the **dose–response curve with a σ_seed band and the prediction line
overlaid** (the headline figure), read the measured-vs-predicted gap, and test the
trimmed-loss defense (recovery fraction ρ with a delta-method CI, the trim-on-clean control,
q-sensitivity, and the kept-vs-dropped accompaniment-energy telemetry). The training/test
cells are **RUN LATER** (GPU); every analysis function is `singnet.analysis.bleed`, exercised
on synthetic data here so the logic is visible before any GPU spend.

The notebook ships **un-executed**. Runtimes on the RUN-LATER banners are from MASTER_PLAN §6.

### State of this notebook
- **Contract:** [`../MASTER_PLAN.md`](../MASTER_PLAN.md) §2 (pre-registration, recapped
  verbatim below), §3.3 (run matrix), §5 (protocol), §6 (budget), §12 (interpretation);
  [`../THEORY.md`](../THEORY.md) §3 (prediction line), §5 (mechanism), §6 (statistics).
- **Data prep is *not* repeated here** — reused from Direction 01
  ([`../../01-loss-function-study/notebooks/01_data_and_eda.ipynb`](../../01-loss-function-study/notebooks/01_data_and_eda.ipynb)).
- **Code, not prose, is authoritative:** the curve/gap/recovery come from
  `singnet.analysis.bleed`; σ_seed from `singnet.analysis.pooled_seed_sigma`; runs from
  `scripts/run_sweep.py --direction 06`.

In [ ]:
# === Colab bootstrap — RUN THIS LATER (Colab only) ==========================
# ⚠️ RUN THIS LATER · ~2 min · CPU.
# !git clone https://github.com/SeanSalvador2/vocal-separation-research.git
# %cd vocal-separation-research
# !pip install -r requirements.txt
# from google.colab import drive; drive.mount('/content/drive')
# import os; os.environ["SHARD_ROOT"] = "/content/drive/MyDrive/musdb_shards"
print("Bootstrap cell — run on Colab only (see comments). No-op here.")

## 1 · Pre-registration recap (MASTER_PLAN §2, verbatim)

> Notation. $s(\varepsilon)$ = mean best-checkpoint **clean** validation vocals SI-SDR of
> the arm trained at bleed level ε. σ_seed = pooled between-seed std over the three
> 3-seed cells (ε = 0, ε = 0.30, trimmed@0.30). $P(\varepsilon)$ = the **prediction
> line**: per validation track, SI-SDR(v + εa, v) computed analytically from the clean
> stems (the score of the estimator that leaks exactly ε of the accompaniment), averaged
> over the 14 tracks; $P(\varepsilon) \approx 10\log_{10}\frac{\lVert v\rVert^2}{\varepsilon^2\lVert a\rVert^2}$
> per track — a −20 log₁₀ε line anchored by each track's vocal/accompaniment energy
> ratio (exact form in THEORY §3).
>
> **H-06a (dose–response).** Bleed hurts, monotonically and detectably:
> - **Supported** iff $s(0) - s(0.30) > \max(\sigma_{\text{seed}}, 0.5\text{ dB})$
>   and the four points are non-increasing in ε within ±σ_seed tolerance.
> - **Refuted (robust)** iff $s(0) - s(0.30) \le \sigma_{\text{seed}}$ — the model
>   shrugs off 30 % bleed (pre-registered as a *surprising positive* about mask-model
>   robustness, immediately checked against the prediction line, which would then sit
>   far below the measured curve).
> - **Mixed** otherwise (non-monotone middle beyond tolerance → investigate before
>   interpreting).
>
> **Shape read-out (descriptive, pre-registered):** the measured curve is compared to
> $P(\varepsilon)$: measured ≈ prediction (±σ_seed band) ⇒ the model faithfully learns
> the corrupted conditional (no implicit denoising); measured **above** ⇒ implicit
> robustness (architecture/loss/augmentation partially reject bleed); measured
> **below** ⇒ corruption additionally destabilizes optimization. This comparison — not
> the bare slope — is the scientific payload.
>
> **H-06b (cheap defense).** At the worst level, trimmed loss recovers a detectable
> fraction:
> - **Supported** iff $s_{\text{trim}}(0.30) - s(0.30) > \sigma_{\text{seed}}$.
>   Report the **recovery fraction**
>   $\rho = \frac{s_{\text{trim}}(0.30) - s(0.30)}{s(0) - s(0.30)}$ with a
>   seed-propagated CI (defined only under H-06a support; else "not evaluable").
> - **Refuted** iff the delta ≤ σ_seed (trimming buys nothing here — a clean negative
>   against the small-loss trick's transfer to *uniform* corruption).
>
> **Control (pre-registered):** trimming at ε = 0 must not cost more than σ_seed on
> clean data; if it does, the defense's price is part of the headline.
>
> **Descriptive secondaries:** q-sensitivity (10 % vs 30 % trim); which chunks get
> trimmed (logged accompaniment-energy statistics of kept vs dropped chunks — THEORY §5
> predicts trimming selects low-⟨a⟩-energy chunks under uniform bleed, the mechanism by
> which small-loss can work at all here); training-curve shapes (Arpit-style late
> divergence between clean and corrupted arms).

## 2 · Run matrix + shared-cell accounting (§3.3)

10 new REDUCED runs; the **clean (ε=0) cell is the shared Direction-01 `l1mag` cell**
across all three clean seeds (config-hash-equal to `base.yaml`; reused, not retrained). The
dry-run below prints ε/q and the eval-guard status per config (CPU, no GPU).

In [ ]:
# CPU-runnable now: the config matrix (ε / q / eval-guard / hash), and the shared-cell check.
from singnet.utils.config import resolve_config, hash_config, corruption_epsilon, trim_q

D06 = '06-robust-training/configs'
D01 = '01-loss-function-study/configs/l1mag_seed0_reduced.yaml'
matrix = ['bleed05_seed0', 'bleed15_seed0', 'bleed30_seed0', 'bleed30_seed1', 'bleed30_seed2',
          'trim30_seed0', 'trim30_seed1', 'trim30_seed2', 'trim30_q10_seed0', 'trim_clean_seed0']
print(f"{'config':<20} {'arm':<11} {'loss':<6} {'ε':<5} {'q':<5} hash")
for n in matrix:
    cfg = resolve_config(f'{D06}/{n}.yaml')
    print(f"{n:<20} {cfg['arm']:<11} {str(cfg.get('loss','l1mag')):<6} "
          f"{str(corruption_epsilon(cfg)):<5} {str(trim_q(cfg)):<5} {hash_config(cfg)}")
base_h = hash_config(resolve_config(f'{D06}/base.yaml'))
print(f"\nclean cell (base.yaml) hash = {base_h}")
print(f"D01 l1mag_seed0_reduced hash = {hash_config(resolve_config(D01))}")
print("shared clean cell (ε=0 ≡ D01, reused not retrained):",
      base_h == hash_config(resolve_config(D01)))

In [ ]:
# ⚠️ RUN THIS LATER (GPU) — the 10 new runs. §6 budget: 10 × REDUCED ≈ 13–18 T4-h
# (~1.3–1.8 h each). Fully resumable; a completed run is skipped by config hash.
#
#   !python scripts/run_sweep.py --direction 06 --dry-run          # CPU: ε/q/eval-guard per config
#   !python scripts/run_sweep.py --direction 06 --stage reduced    # the 10 runs (RUN LATER)
#   # contingency (budget-gated; only if Direction 01 flips the default loss, +2 runs ~3 h):
#   !python scripts/run_sweep.py --direction 06 --stage contingency
print('Launch cells — RUN LATER (GPU). See MASTER_PLAN §6 run book.')

In [ ]:
# ⚠️ RUN THIS LATER (CPU-minutes, needs decoded shards) — the prediction line P(ε) from
# CLEAN validation stems (independent of any run; the §7 G3 audit relies on this).
#
#   !python -m singnet.analysis.bleed --split valid --epsilons 0.05 0.15 0.30 \
#         --shard-root $SHARD_ROOT --out 06-robust-training/results/prediction_valid.csv
print('Prediction line — RUN LATER (needs clean shards).')

## 3 · Headline figure — dose–response curve with σ_seed band **and** the prediction line overlaid

The scientific payload (MASTER_PLAN §2, THEORY §3): the measured clean-validation curve
$s(\varepsilon)$ with its between-seed band, overlaid on the closed-form prediction line
$P(\varepsilon)$. The measured curve sitting **on** $P$ ⇒ faithful learning of the corrupted
conditional; **above** ⇒ implicit robustness; **below** ⇒ optimization damage. The cell
assembles the figure from `singnet` functions; the values below are a **synthetic
illustration** of the layout — the RUN-LATER version swaps in the registry means (measured)
and the shard-derived $P(\varepsilon)$.

In [ ]:
# CPU-runnable now (synthetic illustration of the headline figure layout).
# RUN LATER: replace `measured`/`sigma_seed` with the registry 3-seed means and
# `pred` with prediction_line(clean valid stems). All assembly is singnet + numpy/plt.
import numpy as np
import matplotlib.pyplot as plt
from singnet.analysis import prediction_line, predicted_curve, pooled_seed_sigma

eps_grid = [0.0, 0.05, 0.15, 0.30]

# --- prediction line P(ε) from stems (synthetic here; clean valid stems RUN LATER) ---
rng = np.random.default_rng(3)
stems = [(f't{i}', rng.standard_normal(4096), rng.standard_normal(4096)) for i in range(14)]
pred_df = prediction_line(stems, [e for e in eps_grid if e > 0])
pred = predicted_curve(pred_df)  # indexed by ε>0

# --- measured s(ε) with a σ_seed band (SYNTHETIC illustration) ---
# RUN LATER: read from 06-robust-training/results/registry.csv (3-seed means at ε∈{0,0.30}).
measured = {0.0: 6.2, 0.05: 5.9, 0.15: 5.1, 0.30: 4.0}
seed_cells = {0.0: [6.1, 6.3, 6.2], 0.30: [3.8, 4.1, 4.1]}          # the two anchor cells
sigma_seed = pooled_seed_sigma(seed_cells[0.0], seed_cells[0.30])

fig, ax = plt.subplots(figsize=(7.5, 4.5))
xs = list(measured.keys()); ys = [measured[e] for e in xs]
ax.plot(xs, ys, 'o-', color='#2b6cb0', lw=2, label='measured s(ε)  [clean val, 3-seed mean]')
ax.fill_between(xs, [y - sigma_seed for y in ys], [y + sigma_seed for y in ys],
                color='#2b6cb0', alpha=0.18, label=f'±σ_seed ({sigma_seed:.2f} dB)')
ax.plot(pred.index, pred.values, 's--', color='#c0392b', lw=1.8,
        label='prediction line P(ε)  [−20·log10 ε, from clean stems]')
ax.set_xlabel('bleed level ε'); ax.set_ylabel('clean vocals SI-SDR (dB)')
ax.set_title('Dose–response curve with σ_seed band + prediction-line overlay (SYNTHETIC demo)')
ax.legend(fontsize=8); ax.grid(alpha=0.3); fig.tight_layout(); plt.show()
print('gap read-out (measured − predicted): above ⇒ implicit robustness, below ⇒ optimization damage')

## 4 · Measured-vs-predicted gap per ε

In [ ]:
# CPU-runnable now (synthetic): the per-ε gap table. RUN LATER: real measured + P(ε).
from singnet.analysis import measured_vs_predicted
measured_pos = {0.05: 5.9, 0.15: 5.1, 0.30: 4.0}   # RUN LATER: registry means
gap = measured_vs_predicted(measured_pos, pred)      # pred from the headline cell
print(gap.to_string(index=False))
print('\n> 0 ⇒ measured ABOVE prediction (implicit robustness); < 0 ⇒ BELOW (optimization damage).')

## 5 · The trimmed-loss defense — recovery fraction ρ (H-06b) + the clean control

In [ ]:
# CPU-runnable now (synthetic): recovery fraction ρ with the delta-method CI + control.
# RUN LATER: s_clean/s_bleed/s_trim = 3-seed means; σ's = per-cell between-seed std.
from singnet.analysis import recovery_fraction, pooled_seed_sigma

clean = [6.1, 6.3, 6.2]; bleed = [3.8, 4.1, 4.1]; trim = [4.7, 5.0, 4.9]      # RUN LATER: registry
import numpy as np
r = recovery_fraction(np.mean(clean), np.mean(bleed), np.mean(trim),
                      sigma_clean=np.std(clean, ddof=1), sigma_bleed=np.std(bleed, ddof=1),
                      sigma_trim=np.std(trim, ddof=1), n=3)
print(f'recovery ρ = {r.rho:.3f}   95% CI [{r.ci_lo:.3f}, {r.ci_hi:.3f}]   evaluable={r.evaluable}')
print(f'recovered = {r.recovered_db:.2f} dB of a {r.degradation_db:.2f} dB degradation')

# Control (pre-registered): trimming on CLEAN data must not cost more than σ_seed.
sigma_seed = pooled_seed_sigma(clean, bleed, trim)
trim_clean_s, clean_s = 6.05, np.mean(clean)                                  # RUN LATER: registry
cost = clean_s - trim_clean_s
print(f'\ntrim-on-clean control: cost = {cost:.2f} dB vs σ_seed = {sigma_seed:.2f} dB '
      f'-> {"fires (report the price)" if cost > sigma_seed else "within noise (free)"}')

## 6 · q-sensitivity + the trim telemetry (THEORY §5 signature)

Which chunks get trimmed? The loop logs, every 500 steps, the accompaniment-energy of kept
vs dropped chunks (`trim_energy_stats.csv`). THEORY §5 predicts **kept ⟨a⟩-energy < dropped
⟨a⟩-energy** — the curriculum-by-cleanliness signature. No separation ⇒ the selection signal
never existed ⇒ H-06b refuted with a mechanism. (Direction 08's silence/energy machinery is
the same per-chunk accompaniment-energy accounting, reused here.)

In [ ]:
# ⚠️ RUN THIS LATER — read the per-run telemetry CSVs and compare to the §5 signature.
#   import pandas as pd
#   tele = pd.read_csv('checkpoints/<trim30_hash>/trim_energy_stats.csv')
#   # plot kept_energy_mean vs dropped_energy_mean over steps; the §5 prediction is
#   # kept < dropped throughout. q-sensitivity: overlay trim30 (q=0.30) vs trim30_q10 (q=0.10).
#
# CPU illustration of the expected signature on a constructed batch (see notebook 01 §5):
import torch
from singnet.losses import TrimmedLoss, build
losses = [0.9, 0.1, 0.7, 0.2, 0.5, 0.3, 0.8, 0.4]
mask = torch.zeros(8, 4, 4); mix = torch.ones(8, 4, 4); tgt = torch.zeros(8, 4, 4)
for i, cval in enumerate(losses):
    tgt[i] = cval
for q in (0.10, 0.30):
    _, aux = TrimmedLoss(build('l1mag'), q=q)(mask, mix, tgt, chunk_energy=torch.tensor(losses))
    print(f'q={q}: keep {int(aux["n_kept"])}/8  kept⟨a⟩={aux["kept_energy_mean"]:.3f} '
          f'< dropped⟨a⟩={aux["dropped_energy_mean"]:.3f}  (THEORY §5 signature holds)')

## 7 · Single test pass (RUN LATER) — the two pre-registered paired contrasts

In [ ]:
# ⚠️ RUN THIS LATER (GPU-minutes) — score the three 3-seed cells' best checkpoints
# (clean, bleed30, trim30 = 9 checkpoints) on the 50 CLEAN test tracks; per-track CSV;
# bootstrap 95% CI + Wilcoxon on the two pre-registered pairs (bleed30−clean, trim30−bleed30).
# §6 budget: ~1 h incl. prediction lines. ONE pass only (G3), after val freeze.
#
#   !python scripts/evaluate.py --direction 06 --split test
print('Test pass — RUN LATER (GPU). Two pre-registered pairs only (MASTER_PLAN §5, §6.3).')

## 8 · Interpretation branches (MASTER_PLAN §12) — labeled stubs, filled after freeze

Pre-registered readings; the frozen data selects exactly one row per hypothesis. Full prose
goes to `paper/PAPER.md`.

- **[BRANCH A] H-06a supported, measured ≈ prediction** — the model faithfully learns
  whatever target you give it; data quality is a hard constraint, quantified in dB/ε. The
  curve becomes a calibration chart for "how clean must stems be." *(fill: the per-ε gap)*
- **[BRANCH B] H-06a supported, measured ABOVE prediction** — implicit robustness
  (architecture/loss/augmentation partially reject bleed). Localize the source (remix?
  sigmoid mask cap?) as named future work. *(fill: the above-gap in dB)*
- **[BRANCH C] H-06a supported, measured BELOW prediction** — corruption also destabilizes
  training (damage beyond the information limit). Check curves/grad norms; GCE-style robust
  loss becomes the follow-up. *(fill: the below-gap + grad-norm evidence)*
- **[BRANCH D] H-06a refuted (robust/flat)** — 30 % bleed within noise; verify P(ε) sits
  well below the measured curve (a surprising, publishable positive). *(fill: s(0)−s(0.30) vs σ_seed)*
- **[BRANCH E] H-06b supported (ρ>0 detectably)** — the small-loss trick transfers to
  uniform bleed via energy-selection; adopt trimming as an optional flag. *(fill: ρ + CI + the §5 telemetry)*
- **[BRANCH F] H-06b refuted** — trimming fails without sample-level cleanliness; an honest
  boundary for the classic trick. *(fill: the delta ≤ σ_seed + the flat telemetry)*
- **[BRANCH G] Control fires (trim hurts clean > σ_seed)** — the defense has a price on
  clean data; recommend trimming only under suspected corruption. *(fill: the clean cost)*

## 9 · Conclusions + cross-links

- The **dose–response curve vs the prediction line** turns "how much does bleed hurt" into a
  measured, falsifiable statement — and even a *flat* curve becomes a quantified robustness
  claim (via P(ε) sitting below it).
- The **trimmed-loss defense** is run *outside* its proven sample-level regime; the
  kept-vs-dropped ⟨a⟩-energy telemetry decides *why* it works or fails (curriculum-by-cleanliness
  vs no cleanliness gradient), THEORY §5.
- **Cross-links.** Direction 05's *Why LoRA Resists Label Noise* (arXiv 2602.00084) is a
  natural follow-up bridge — LoRA as an inherently noise-robust adapter under the same
  corruption (future work). Direction 08's silence/energy machinery is the *same* per-chunk
  accompaniment-energy accounting reused by this direction's trim telemetry.